## Step 1: Data Warehouse Modeling: Star Schema


1.1. Define the Business process and the fact grain 

Business Process: The business process consists of analyzing stock trading transactions. The process includes buy and sell operations of stocks and supports analysis by time, company, sector, industry, transaction type, and geography.

Granularity refers to the level of detail stored in the fact table. The grain of the Fact_Transactions table is one row per financial transaction. Each transaction represents a single buy or sell operation of a specific financial instrument on a given date.

1.2 Identify Fact and Dimensions

**FACT TABLE**
+ *Fact_transactions*: IDTransaction, ID_Symbol, ID_Geography, ID_Date, ID_TransactionType, Unit 

**DIMENSION TABLES**

 + *Dim_symbol*: ID_Symbol, symbol,company_name,industry, sector 

 + *Dim_geography*: ID_Geography, name,alpha_2,alpha_3,country-code,iso-3166-2,region, sub-region,intermediate-region, region-code,sub-region-code,intermediate-region-code 

 + *Dim_time*:ID_Date, Date, day, month, quarter, year

 + *Dim_TransactionType*: ID_TransactionType, TransactionType



1.3 Define the hierarchy levels available in each dimension.

Hierarchies define how data can be aggregated and analyzed at different levels of granularity. They allow decision-makers to view facts at different levels of detail.

Dimension Hierarchies
+ Dim_time: day -> month -> quarter -> year
+ Dim_geography: country -> sub-region -> region
+ Dim_symbol: symbol -> industry -> sector
+ Dim_TransactionType: No hierarchy (flat dimension)


Fact_Transactions contains the measure Unit, the foreign keys linking each transaction to the dimension tables, and the transaction identifier (IDTransaction), which identifies each transaction.

**FACT TABLE**

+ IDTransaction 
+ ID_Symbol (Foreign key → Dim_symbol)
+ ID_Date (Foreign key → Dim_time)
+ ID_Geography (Foreign key -> Dim_geography)
+ ID_TransactionType (Foreign key → Dim_TransactionType)
+ Unit (measure) 

**DIMENSION TABLES**

 + *Dim_symbol*: ID_Symbol (surrogate primary key),symbol,company_name,industry,sector

 + *Dim_geography*: ID_Geography (surrogate primary key),name,alpha_2,alpha_3,country-code,iso3166-2,region, sub-region,intermediate-region, region-code,sub-region-code,intermediate-region-code 

 + *Dim_time*: ID_Date (surrogate primary key), Date, day,month, quarter, year

 + *Dim_TransactionType*: ID_TransactionType (surrogate primary key), TransactionType


**Descriptive variables**

 + *Dim_symbol*: symbol,company_name,industry,sector --> these columns provide more information about the company features

 + *Dim_geography*:country,alpha_2,alpha_3,country-code,iso3166-2,region, sub-region,intermediate-region, region-code,sub-region-code,intermediate-region-code -> These      attributes describe the geographical location of the issuing company

 + *Dim_time*: Date, day,month, quarter, year

 + *Dim_TransactionType*: TransactionType  --> It defines the type of transaction (BUY or SELL).
                     


## Part 2 Data transformation and analysis 

In [1]:
# 2.1  Create the Dataframe required by star schema. Keep in each dataframe only the information included in your dimensional model 

import pandas as pd

df = pd.read_csv('data/account-statement-1-1-2024-12-31-2024.csv', sep=';')
df_1 = pd.read_csv('data/country.csv')
df_2 = pd.read_csv('data/symbols.csv',sep=';')

In [2]:
#Cheking missing values

print(df.isna().sum())

#The original dataset was highly affected by missing values; 
#therefore, records with incomplete data were removed during the preprocessing stage to obtain a cleaner and more reliable dataset for analysis.

df = df.dropna(subset=[
    "IDTransaction",
    "Date",
    "TransactionType",
    "Symbol",
    "Unit"
]).reset_index(drop=True)

IDTransaction       464
Date                464
TransactionType     464
Symbol              464
Unit                464
Unnamed: 5         2745
dtype: int64


In [3]:
# I check only the attributes belonging to geography_dim, excluding the unused attributes.

# In this case, the original dataset (df_1) shows a high level of data quality, as there are only two missing values in the "region" and "sub-region" attributes.
# Therefore, removing observations with missing values is not necessary.

print(df_1[[ 'country-code','name','region','alpha-3','sub-region']].isna().sum())


country-code    0
name            0
region          2
alpha-3         0
sub-region      2
dtype: int64


In [4]:
# Within the orginal dataset (df_2) there are not missing values 

print(df_2.isna().sum())

symbol          0
company_name    0
sector          0
industry        0
country         0
dtype: int64


In [5]:
#removing unnecessary spaces to ensure consistent joins between tables.

df = df.copy()
df_2 = df_2.copy()
df_1 = df_1.copy()

df.columns = df.columns.str.strip()
df_2.columns = df_2.columns.str.strip()
df_1.columns = df_1.columns.str.strip()

df["Symbol"] = df["Symbol"].astype(str).str.strip().str.upper()
df_2["symbol"] = df_2["symbol"].astype(str).str.strip().str.upper()

In [6]:
#Create the dimension tables  and the respectively surrogate keys 

# TRANSACTIONTYPE DIMENSION
dim_TransactionType = df[['TransactionType']].drop_duplicates().reset_index(drop=True)
dim_TransactionType.insert(0, 'ID_TransactionType', range(1, len(dim_TransactionType) + 1)) # surrogate key (ID_TransactionType)


# SYMBOL DIMENSION 
dim_symbol = df_2[['symbol','company_name','industry','sector']].drop_duplicates().reset_index(drop = True)
dim_symbol.insert(0, 'ID_Symbol', range(1, len(dim_symbol) + 1)) # surrogate key (ID_Symbol)


# GEOGRAPHY DIMENSION 
dim_geography = df_1[[ 'country-code','name','region','alpha-3','sub-region']].dropna(subset=['region','sub-region']).drop_duplicates().reset_index(drop=True)
dim_geography.insert(0,'ID_Geography', range(1,len(dim_geography)+1)) #surrogate key (ID_Country)


# TIME GEOGRAPHY 
df['Date'] = pd.to_datetime(df['Date'], dayfirst=True)

dim_time = df[['Date']].drop_duplicates().reset_index(drop=True)

dim_time['day'] = dim_time['Date'].dt.day.astype('Int64') #create day column
dim_time['month'] = dim_time['Date'].dt.month.astype('Int64') # create month column
dim_time['quarter'] = dim_time['Date'].dt.to_period('Q').astype(str) # create quarter column
dim_time['year'] = dim_time['Date'].dt.year.astype('Int64') # create year column 

dim_time.insert(0, 'ID_Date',range(1, len(dim_time) + 1)) # surrogate key (ID_Date)



In [7]:
# Create the fact table

fact_table = df.copy()

# PS: left join is useful to  preserve all records in the fact table, even if no matching value exists in the dimension table.

#A left join was performed between the fact table and the symbol dimension
# to retrieve the corresponding surrogate key (ID_Symbol) for each value of the Symbol attribute.

fact_table = fact_table.merge(
    dim_symbol[['symbol', 'ID_Symbol']],
    left_on='Symbol',
    right_on='symbol',
    how='left'
)

#A left join was performed between the fact table and the TransactionType dimension
# to add the corresponding surrogate key (ID_TransactionType) for each value of the TransactionType attribute.

fact_table = fact_table.merge(
    dim_TransactionType,
    on='TransactionType',
    how='left'
)


#A left join was performed between the fact table and the time dimension
# to add the corresponding surrogate key (ID_Date) for each value of the Date attribute.

fact_table = fact_table.merge(
    dim_time[['Date', 'ID_Date']],
    on='Date',
    how='left'
)

# merge between fact_table and the original dataset (df_2) to add into the fact table the country attribute 
# otherwise it is not possibile to create a connection between the fact table and dim_geography. 

fact_table = fact_table.merge(
    df_2[['symbol', 'country']],
    left_on='Symbol',
    right_on='symbol',
    how='left'
)

# merge between fact_table and dim_geography

fact_table = fact_table.merge(
    dim_geography[['name', 'ID_Geography']],
    left_on='country',
    right_on='name',
    how='left'
)

# Select only the relevant columns to define the final structure of the fact table,
# keeping the natural key (IDTransaction), the  surrogate keys and the measure (Unit).

fact_table = fact_table[[
    'IDTransaction',
    'ID_Date',
    'ID_Symbol',
    'ID_TransactionType',
    'ID_Geography',
    'Unit'
]]


# Check missing values within the fact table 

print("---check missing values---")
print(fact_table.isna().sum())
print(fact_table.head())

---check missing values---
IDTransaction           0
ID_Date                 0
ID_Symbol             212
ID_TransactionType      0
ID_Geography          306
Unit                    0
dtype: int64
   IDTransaction  ID_Date  ID_Symbol  ID_TransactionType  ID_Geography    Unit
0   2.769834e+09        1      284.0                   1         174.0  1605.0
1   2.767325e+09        2      284.0                   2         174.0  1605.0
2   2.815474e+09        3      284.0                   2         174.0   914.0
3   2.622244e+09        4        4.0                   1          24.0   646.0
4   2.629871e+09        5      258.0                   2         130.0   646.0


In [8]:
#verify that every transaction symbol exists in the symbols dataset;

df_clean = df['Symbol'].dropna().drop_duplicates()
df_2_clean = df_2['symbol'].dropna().drop_duplicates()

def function(df_clean, df_2_clean):
    G = []
    for symbol in df_clean:
        if not symbol in df_2_clean.values:
            G.append(symbol)
    return G

result = function(df_clean, df_2_clean)

if len(result) == 0:
    print("All transaction symbols exist in the symbols dataset.")
else:
    print("The following transaction symbols do not exist in the symbols dataset:")
    print(result)



The following transaction symbols do not exist in the symbols dataset:
['MFG', 'AGO.L', 'TKC', 'FNC', 'HTGC', 'IBE', 'OBDC', 'CCAP', 'AZM', 'VWS', 'RCMT', 'WF', 'UCG', 'MONC', 'RIGZU', 'SAP', 'CSIQ', 'ARCH']


In [9]:
#verify that every company country can be mapped to the country dataset.

df_2_clean_1 = df_2['country'].dropna().drop_duplicates()
df_1_clean = df_1['name'].dropna().drop_duplicates()

def function(df_2_clean_1, df_1_clean):
    F = []
    for country in df_2_clean_1:
        if not country in df_1_clean.values:
            F.append(country)
    return F

result_1 = function(df_2_clean_1, df_1_clean)

if len(result_1) == 0:
    print("All company countries can be mapped to the country dataset.")
else:
    print("The following company countries cannot be mapped to the country dataset:")
    print(result_1)

The following company countries cannot be mapped to the country dataset:
['Taiwan', 'Turkey']


In [10]:
# 1. What are the top 5 sectors by number of SELL transactions in US during 2024?

# sector → dim_symbol
# SELL → dim_transactionType
# USA (alpha-3) → dim_geography
# year (2024) → dim_time

# The dimensions are joined with the fact table using surrogate keys.

# WHERE: defines the conditions (SELL transactions, USA, year 2024)

# COUNT(*) + GROUP BY s.sector: counts the number of transactions for each sector

# ORDER BY num_transactions DESC: sorts results from highest to lowest number of transactions

# LIMIT 5: returns the top 5 sectors with the highest number of transactions

import duckdb

result = duckdb.query("""
    SELECT  
        s.sector,
        COUNT(*) AS num_SELL_transactions
    FROM fact_table f
    JOIN dim_symbol s ON f.ID_Symbol = s.ID_Symbol
    JOIN dim_TransactionType t ON f.ID_TransactionType = t.ID_TransactionType
    JOIN dim_geography g ON f.ID_Geography = g.ID_Geography
    JOIN dim_time d ON f.ID_Date = d.ID_Date
    WHERE  
        t.TransactionType = 'SELL'
        AND g."alpha-3" = 'USA'
        AND d.year = 2024
    GROUP BY
        s.sector
    ORDER BY
        num_SELL_transactions DESC
    LIMIT 5
""").df()

print(result)

                   sector  num_SELL_transactions
0              Technology                    158
1  Communication Services                     58
2      Financial Services                     55
3              Healthcare                     50
4       Consumer Cyclical                     48


In [11]:
#2. What are the top 5 industries by number of BUY transactions in Q4 of 2024?

# industry → dim_symbol
# BUY → dim_transactionType
# quarter (2024Q4)-> dim_time

result_1 = duckdb.query("""
    SELECT 
        s.industry,
        COUNT(*) AS num_BUY_transactions
    FROM fact_table f
    JOIN dim_symbol s ON f.ID_Symbol = s.ID_Symbol
    JOIN dim_TransactionType t ON f.ID_TransactionType = t.ID_TransactionType
    JOIN dim_time d ON f.ID_Date = d.ID_Date
    WHERE  
        t.TransactionType = 'BUY'
        AND  d.quarter = '2024Q4'
    GROUP BY
        s.industry
    ORDER BY
        num_BUY_transactions DESC
    LIMIT 5
""").df()

print(result_1)

                         industry  num_BUY_transactions
0                  Semiconductors                    18
1  Internet Content & Information                    15
2       Software - Infrastructure                    10
3                 Internet Retail                     8
4          Diagnostics & Research                     7


In [12]:
#3. What are the top 3 sectors by total number of units sold on Mondays across 2024?
#sector -> dim_symbol 
#unit -> fact_table 
#SELL  -> dim_TransactionType
#Mondays -> dim_time
#2024 -> dim_time

# we add a new attribute within time dimension in order to resolve the query 
dim_time['day_name']= dim_time['Date'].dt.day_name()

result_3 = duckdb.query("""
    SELECT 
        s.sector,
        SUM(f.Unit) AS Total_Units
    FROM fact_table f
    JOIN dim_symbol s ON f.ID_Symbol = s.ID_Symbol
    JOIN dim_TransactionType t ON f.ID_TransactionType = t.ID_TransactionType
    JOIN dim_time d ON f.ID_Date = d.ID_Date
    WHERE  
        t.TransactionType = 'SELL'
        AND d.day_name = 'Monday'
        AND d.year = 2024
    GROUP BY
        s.sector
    ORDER BY
        Total_Units DESC
    LIMIT 3
""").df()

print(result_3)

              sector  Total_Units
0         Technology       4361.0
1         Healthcare       2108.0
2  Consumer Cyclical       1218.0


In [13]:
#4.  What are the top 5 regions by number of distinct industries traded?

result_4= duckdb.query("""
    SELECT 
        g.region,
        IFNULL(COUNT(DISTINCT s.industry),0) AS num_industries
     FROM dim_geography g
    LEFT JOIN fact_table f  ON f.ID_Geography = g.ID_Geography
    LEFT JOIN dim_symbol s ON f.ID_Symbol = s.ID_Symbol
    GROUP BY
        g.region
    ORDER BY
        num_industries DESC
    LIMIT 5
""").df()

print(result_4)

     region  num_industries
0  Americas              38
1    Europe              26
2      Asia               9
3    Africa               0
4   Oceania               0


In [14]:
# 5. What are the top 10 symbols by number of transactions (BUY + SELL) in 2024?
#symbols -> dim_symbol 
#2024 - > dim_time

import duckdb

result_5 = duckdb.query("""
    SELECT  
        s.symbol,
        COUNT(*) AS num_transactions
    FROM fact_table f
    JOIN dim_symbol s ON f.ID_Symbol = s.ID_Symbol
    JOIN dim_time d  ON f.ID_Date = d.ID_Date
    WHERE d.year = 2024
    GROUP BY s.symbol
    ORDER BY num_transactions DESC
    LIMIT 10
""").df()

print(result_5)


  symbol  num_transactions
0    ARM               100
1    AMD                97
2    TSM                80
3   TIMB                76
4   GOOG                52
5   MSFT                49
6   AMZN                47
7   ARDX                43
8    BLK                42
9   BRFS                42


## Summary


First, I redesigned the three datasets into a star schema to improve data organization and simplify querying. The star schema consists of a central fact table connected to multiple dimension tables, enabling efficient aggregation and reporting.

Next, I performed a data quality assessment by identifying missing values within each dataset. After cleaning the transaction dataset, I created the dimension tables and assigned surrogate keys to ensure consistent and efficient relationships between entities.Finally, I built the fact table, which contains the measures and includes the foreign keys linking it to each dimension table. 

To extract more information about the datasets, I developed two verification functions: the first checks whether each transaction symbol exists in the symbols dataset, while the second verifies whether each company country can be mapped to the country dataset.The results revealed that:
+ 18 transaction symbols could not be matched to any company in the symbols dataset.
+ Two countries, Taiwan and Turkey, could not be mapped to the country dataset.

Furthermore, five analytical queries were performed to extract key insights from the data:

1. **Top sectors by number of sell transactions in 2024**: Technology (158), Communication Services (58), Financial Services (55), Healthcare (50), and Consumer Cyclical (48).

2. **Top industries by number of buy transactions in Q4 2024**: Semiconductors (18), Internet Content & Information (15), Software – Infrastructure (10), Internet Retail (8), and Diagnostics & Research (7).

3. **Top sectors by total units sold on Mondays during 2024**: Technology (4.361), Healthcare (2.108), and Consumer Cyclical (1.218).

4. **Regions ranked by number of distinct industries traded**: Americas (38), Europe (26), Asia (9), Oceania (0), and Africa (0).
 To include regions with no recorded trades, a left join was applied.

5. **Top 10 stock symbols by number of transactions in 2024**: ARM (100), AMD (97), TSM (80), TIMB (76), GOOG (52), MSFT (49), AMZN (47), ARDX (43), BRFS (42), and BLK (42).

Finally, I implemented an interactive dashboard using Streamlit.The dashboard allows users to explore the data through interactive charts and filters, providing a more intuitive way to verify trends. 